# MMAF-CD v25 — Stage F2: explanation benchmark

Re-uses the Stage F setup (same e_table, gateway, 16 v25 specialists) and then tests **every explanation
method the thesis describes**, implemented correctly, with the same three tests on the same images:

| test | what it shows |
|---|---|
| **faithfulness** — deletion / insertion AUC vs a random pixel order (Wilcoxon, 40 test images per organ) | does the map rank the pixels the model actually uses? |
| **sanity** — model-randomisation test (Adebayo et al. 2018) | does the map depend on the trained weights at all? |
| **localisation** — Dice, pointing game, energy-in-mask, map-box → MedSAM (skin, kidney, esophageal, thyroid masks) | does it point at the lesion? |
| **trust signals** — AUROC for detecting the specialist's own errors | does confidence / MC-dropout / TTA / CAM focus predict a wrong answer? |

Methods: Grad-CAM · Grad-CAM++ (correct) · **Grad-CAM++ exactly as in the thesis code** (Σ A·g³ denominator, score×100) ·
LayerCAM (stage 3, 14×14) · EigenCAM · Score-CAM (top-64 channels) · Integrated Gradients (blur baseline, 32 steps) ·
RISE (800 masks) · random.

**Standalone & lean** — no old models, no old gateway, no 25-min base pipeline; only the inputs below.
Accelerator GPU T4 ×2 · Internet ON (MedSAM) · ~1.5–2 h · output `mmaf_v25_F2.zip`.
A dataset that is not attached (or fails to mount) only shrinks the sample — the coverage table printed by the
first cell shows exactly which test images were readable.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# F2-0 — standalone setup (no old models, no old gateway, no 25-min base pipeline)
#   • finds the v25 table + 16 specialists in the attached mmaf-v25-e2 / e2p datasets
#   • re-maps image paths if Kaggle mounted a dataset at a different path than in the E2 run
#   • rebuilds the pancreas crop rows from msd-pancreas-crops (same split as E2P/F)
#   • rebuilds KiTS23 PNGs on demand (E2/F wrote them to /kaggle/working at run time)
#   • every image is checked lazily → a dataset that is missing just shrinks the sample, it never crashes
# ════════════════════════════════════════════════════════════════════════════════
import os, re, gc, glob, json, math, time, copy, random, hashlib, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2, timm, torch, torch.nn as nn
from PIL import Image
warnings.filterwarnings("ignore"); Image.MAX_IMAGE_PIXELS = None
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
T0 = time.time(); TIME_BUDGET_H = 11.0
def e_elapsed_h(): return (time.time() - T0) / 3600
def free_mem(tag=""):
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"device={DEVICE}  gpus={torch.cuda.device_count()}  torch={torch.__version__}  timm={timm.__version__}")

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXT_SPLIT = (0.60, 0.10, 0.30)
E2_BACKBONE = "convnext_small.fb_in22k_ft_in1k"
IMN_MEAN = np.array([0.485, 0.456, 0.406], np.float32); IMN_STD = np.array([0.229, 0.224, 0.225], np.float32)
MALIGNANT_CLASSES = {
    "blood": {"early", "pre", "pro"}, "lymphoma": {"cll", "fl", "mcl"},
    "skin": {"actinic_keratosis", "basal_cell_carcinoma", "melanoma", "squamous_cell_carcinoma"},
    "cervical": {"dyk", "koc", "mep", "pab"}, "colon": {"adenocarcinoma"}, "brain_2d": {"glioma", "meningioma", "tumor"},
    "esophageal": {"malignant"}, "kidney": {"tumor"}, "lung": {"malignant"}, "oral": {"scc"}, "pancreatic": {"tumor"},
    "thyroid": {"malignant"}, "breast": {"malignant"}, "bladder": {"muscle_invasive", "non_muscle_invasive"},
    "gastric": {"tumor", "deb", "lym", "str"},
    "ovarian": {"clear_cell", "endometrioid", "high_grade_serous", "low_grade_serous", "mucinous"}}

# ---- image helpers (identical to Stage E/F) -----------------------------------------------------
def load_rgb(p, max_side=768):
    p = ensure(p); im = Image.open(p)
    try: im.draft("RGB", (max_side, max_side))
    except Exception: pass
    if im.mode in ("I;16", "I;16B", "I"):
        a = np.asarray(im, dtype=np.float32); a = (a - a.min()) / (a.max() - a.min() + 1e-6) * 255
        im = Image.fromarray(a.astype(np.uint8))
    a = np.asarray(im.convert("RGB")); s = max(a.shape[:2])
    if s > max_side:
        f = max_side / s; a = cv2.resize(a, (max(1, int(a.shape[1] * f)), max(1, int(a.shape[0] * f))), interpolation=cv2.INTER_AREA)
    return np.ascontiguousarray(a)
def to_tensor(img):
    x = (img.astype(np.float32) / 255. - IMN_MEAN) / IMN_STD
    return torch.from_numpy(np.ascontiguousarray(x.transpose(2, 0, 1)))
def eval_img(img, size=224): return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
def list_images(root): return sorted(str(p) for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT and p.is_file())
def _u01(s): return int(hashlib.md5(s.encode()).hexdigest()[:12], 16) / float(16 ** 12)
def _assign(source, group, fr):
    u = _u01(f"{source}|{group}"); return "train" if u < fr[0] else ("val" if u < fr[0] + fr[1] else "test")

# ---- attached inputs: search only small inputs, never walk the big image datasets ----------------
_BIG = ("multi-cancer", "cbis-ddsm", "gastric-cancer-data", "cancer-inst-segmentation", "kits23", "isic2018", "leukemia",
        "pmram", "lunghist700", "crc-val", "tn3k", "ddti", "edd2020", "pancreasct", "skin-cancer", "mendeley-lbc",
        "ovarian-cancer", "monuseg", "/models/")
def _roots(big=False):
    out = []
    for pat in ("/kaggle/input/datasets/*/*", "/kaggle/input/notebooks/*/*", "/kaggle/input/*"):
        for r in glob.glob(pat):
            if Path(r).name in ("datasets", "notebooks", "models", "competitions") or not Path(r).is_dir(): continue
            if not big and any(b in r.lower() for b in _BIG): continue
            out.append(r)
    return sorted(set(out))
def find_all(pattern, maxdepth=5):
    hits = []
    for r in _roots():
        for d in range(0, maxdepth + 1): hits += glob.glob(r + "/*" * d + "/" + pattern)
    return sorted(set(hits))
def find_slug(slug):
    for pat in (f"/kaggle/input/datasets/*/{slug}", f"/kaggle/input/{slug}", f"/kaggle/input/notebooks/*/{slug}"):
        h = [x for x in glob.glob(pat) if Path(x).is_dir()]
        if h: return h[0]
    return None
print("attached inputs:"); [print("   ", r) for r in _roots(big=True)]

# ---- table ------------------------------------------------------------------------------------
_tabs = find_all("e_table.csv")
assert _tabs, "❌ e_table.csv not found — attach the mmaf-v25-e2 dataset"
_n = {t: sum(1 for _ in open(t, errors="ignore")) for t in _tabs}
E_PREV_TABLE = max(_n, key=_n.get)
TAB = pd.read_csv(E_PREV_TABLE, usecols=["path", "organ", "source", "kind", "group", "split", "label_set", "spec_ok", "holdout"],
                  dtype={"label_set": str, "group": str}, low_memory=False)
TAB["label_set"] = TAB.label_set.fillna(""); TAB["spec_ok"] = TAB.spec_ok.astype(str).str.lower().eq("true")
print(f"table: {E_PREV_TABLE}  ({len(TAB):,} rows; candidates: {[ (t, _n[t]) for t in _tabs ]})")

# ---- re-map dataset roots whose mount path changed --------------------------------------------
def _root_of(p):
    q = p.split("/")
    if p.startswith(("/kaggle/input/datasets/", "/kaggle/input/notebooks/")) and len(q) > 5: return "/".join(q[:6]), q[5]
    if p.startswith("/kaggle/input/") and len(q) > 3: return "/".join(q[:4]), q[3]
    return None, None
_rt = TAB.path.map(lambda p: _root_of(p)[0]); _moved = {}
for r in _rt.dropna().unique():
    if Path(r).exists(): continue
    new = find_slug(r.rstrip("/").split("/")[-1])
    if new and new != r:
        m = _rt == r; TAB.loc[m, "path"] = new + TAB.loc[m, "path"].str[len(r):]; _moved[r] = new
for a, b in _moved.items(): print(f"   re-mapped {a} → {b}")
_missing_roots = sorted(r for r in _rt.dropna().unique() if not Path(_moved.get(r, r)).exists())
for r in _missing_roots: print(f"   ⚠️ not attached: {r}  ({int((_rt == r).sum()):,} rows will be skipped)")

# ---- pancreas: crops from Stage P1b v2 (as E2P / F) -------------------------------------------
def find_msd_crops():
    for m in find_all("metadata.csv", 4):
        q = Path(m).parent
        if "crop" in q.name.lower() and (q / "tumor").is_dir() and (q / "normal").is_dir(): return q
    return None
_cr = find_msd_crops()
if _cr is not None:
    meta = pd.read_csv(_cr / "metadata.csv"); rows = []
    for src, name in [("msd", "msd_task07"), ("nih", "nih")]:
        d = meta[meta.source == src]
        for p, y, c in zip(d.path, d.label, d.case.astype(str)):
            rows.append({"path": str(_cr / p), "organ": "pancreatic", "source": name, "kind": "external", "group": c,
                         "split": "test" if src == "nih" else _assign(name, c, EXT_SPLIT), "label_set": y, "spec_ok": True, "holdout": src == "nih"})
    TAB = pd.concat([TAB[~((TAB.organ == "pancreatic") & TAB.source.isin(["msd_task07", "nih"]))], pd.DataFrame(rows)], ignore_index=True)
    print(f"pancreas crops: {_cr}  → {len(rows)} rows ({sum(r['split'] == 'test' for r in rows)} test)")
else:
    print("⚠️ msd-pancreas-crops not attached → pancreatic uses the whole-slice rows (the E2P specialist expects crops)")

# ---- KiTS23: PNGs + tumour masks rebuilt on demand --------------------------------------------
EXT_ROOT = Path("/kaggle/working/external"); (EXT_ROOT / "kits_png").mkdir(parents=True, exist_ok=True)
_KITS = find_slug("kits23-2d-kidney-tumor-segmentation"); KITS_IMG = KITS_MSK = None
if _KITS:
    for d in range(0, 4):                                   # literal last component → only directory names are listed
        if KITS_IMG is None: KITS_IMG = next((Path(q) for q in glob.glob(_KITS + "/*" * d + "/images") if Path(q).is_dir()), None)
        if KITS_MSK is None: KITS_MSK = next((Path(q) for q in glob.glob(_KITS + "/*" * d + "/masks") if Path(q).is_dir()), None)
print(f"KiTS23: images={KITS_IMG}  masks={KITS_MSK}")
def _kits_png(name):
    o = EXT_ROOT / "kits_png" / name
    if o.exists() or KITS_IMG is None or not (KITS_IMG / name).exists(): return o
    a = cv2.imread(str(KITS_IMG / name), cv2.IMREAD_UNCHANGED)
    a8 = (a / 257.).astype(np.uint8) if a.dtype == np.uint16 else a.astype(np.uint8)
    cv2.imwrite(str(o), cv2.cvtColor(a8, cv2.COLOR_GRAY2BGR) if a8.ndim == 2 else a8); return o
def _kits_mask(name):
    o = EXT_ROOT / "kits_png" / name.replace(".png", "_tumour.png")
    if not o.exists() and KITS_MSK is not None and (KITS_MSK / name).exists():
        mm = cv2.imread(str(KITS_MSK / name), cv2.IMREAD_UNCHANGED); cv2.imwrite(str(o), ((mm == 2) * 255).astype(np.uint8))
    return o
def ensure(p):
    p = str(p)
    if "/kits_png/" in p and not os.path.exists(p): return str(_kits_png(Path(p).name))
    return p
_EX = {}
def exists(p):
    if p not in _EX:
        try: _EX[p] = os.path.exists(ensure(p))
        except Exception: _EX[p] = False
    return _EX[p]
def first_existing(df, n):
    keep = []
    for i, p in zip(df.index, df.path):
        if exists(p): keep.append(i)
        if len(keep) >= n: break
    return df.loc[keep]
def per_group_existing(df, n, seed, keys=("organ", "source")):
    return pd.concat([first_existing(g, n) for _, g in df.sample(frac=1.0, random_state=seed).groupby(list(keys))] or [df.head(0)])

# ---- specialists (E2; pancreatic from E2P) ----------------------------------------------------
class SEHead(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__(); b = max(1, dim // reduction)
        self.fc = nn.Sequential(nn.Linear(dim, b, bias=False), nn.GELU(), nn.Linear(b, dim, bias=False), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x)
class SpecialistModel(nn.Module):
    def __init__(self, backbone_name, n_classes, dropout=0.25, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0); D = self.backbone.num_features
        self.head = nn.Sequential(nn.Dropout(dropout), SEHead(D), nn.LayerNorm(D), nn.Dropout(dropout / 2), nn.Linear(D, n_classes))
    def forward(self, x): return self.head(self.backbone(x))

_cands = {}
for p in find_all("spec_v25/*.pth"): _cands.setdefault(Path(p).stem, []).append(p)
NEW_SPEC, E_CLASSES = {}, {}
for organ, ps in sorted(_cands.items()):
    best = None
    for p in ps:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        rank = (ck.get("stage", "E2") == "E2P", Path(p).stat().st_mtime)
        if best is None or rank > best[0]: best = (rank, p, ck)
    _, p, ck = best
    m = SpecialistModel(ck.get("backbone_name", E2_BACKBONE), len(ck["class_names"]), dropout=ck.get("dropout", 0.25))
    m.load_state_dict({k: v.float() for k, v in ck["model_state_dict"].items()}); m.eval()
    NEW_SPEC[organ] = {"model": m, "T": float(ck.get("temperature", 1.0)), "stage": ck.get("stage", "E2"), "path": p}
    E_CLASSES[organ] = list(ck["class_names"]); del ck
    print(f"  ✅ {organ:<11} {NEW_SPEC[organ]['stage']:<4} T={NEW_SPEC[organ]['T']:.2f}  {p}")
assert NEW_SPEC, "❌ no spec_v25/*.pth found — attach mmaf-v25-e2 (and mmaf-v25-e2p)"
E_ORGANS = sorted(NEW_SPEC); CLS_IDX = {o: {c: i for i, c in enumerate(E_CLASSES[o])} for o in E_ORGANS}
TAB = TAB[TAB.organ.isin(E_ORGANS)].reset_index(drop=True)
def _sets_of(rows, organ):
    return [",".join(str(CLS_IDX[organ][c]) for c in sorted(s.split("|")) if c in CLS_IDX[organ]) if s else "" for s in rows.label_set]

# ---- coverage report: which test images can actually be read ----------------------------------
print(f"\n{'organ':<11}{'source':<18}{'test rows':>10}{'readable (sample of 20)':>26}")
_cov = {}
for (o, s), g in TAB[TAB.split == "test"].groupby(["organ", "source"]):
    smp = g.sample(min(20, len(g)), random_state=0); r = float(np.mean([exists(p) for p in smp.path]))
    _cov[f"{o}|{s}"] = {"test_rows": int(len(g)), "readable_frac": r}
    print(f"{o:<11}{s:<18}{len(g):>10}{r:>20.0%}" + ("   ⚠️" if r < 0.5 else ""))

# ---- lesion masks for localisation ------------------------------------------------------------
def seg_pairs():
    out = {}
    isic = find_slug("isic2018-challenge-task1-data-segmentation")
    if isic:
        L = list_images(isic); gt = {Path(p).stem.replace("_segmentation", ""): p for p in L if Path(p).stem.endswith("_segmentation")}
        out["skin"] = [(p, gt[Path(p).stem]) for p in L if Path(p).stem in gt and not Path(p).stem.endswith("_segmentation")][:300]
    k = TAB[(TAB.organ == "kidney") & (TAB.source == "kits23") & (TAB.label_set == "tumor")]
    if len(k) and KITS_MSK is not None:
        pr = []
        for p in k.path:
            q = ensure(p)
            if os.path.exists(q):
                mk = _kits_mask(Path(q).name)
                if mk.exists(): pr.append((q, str(mk)))
            if len(pr) >= 300: break
        out["kidney"] = pr
    edd = find_slug("edd2020-endoscopy-detection-and-segmentation")
    if edd:
        rx = re.compile(r"^(.+)_(be|suspicious|hgd|cancer|polyp)(?:_mask)?$", re.I); mask_of, ims = {}, {}
        for p in list_images(edd):
            st = Path(p).stem; m = rx.match(st)
            if m and Path(p).suffix.lower() in (".tif", ".tiff", ".png"):
                if m.group(2).lower() in ("cancer", "hgd"): mask_of.setdefault(m.group(1).lower(), p)
            elif Path(p).suffix.lower() in (".jpg", ".jpeg", ".png"): ims[st.lower()] = p
        out["esophageal"] = [(ims[b], m) for b, m in mask_of.items() if b in ims][:300]
    tn = find_slug("tn3k-thyroid-nodule-region-segmentation-dataset")
    if tn:
        ri = [(p, Path(p).relative_to(tn).as_posix().lower()) for p in list_images(tn)]
        ims = {Path(p).stem: p for p, r in ri if r.split("/")[-2:-1] == ["test-image"]}
        gts = {Path(p).stem: p for p, r in ri if r.split("/")[-2:-1] == ["test-mask"]}
        out["thyroid"] = [(ims[k], gts[k]) for k in ims if k in gts][:300]
    for k, v in out.items(): print(f"   seg pairs {k}: {len(v)}")
    return out
print(f"\n✅ setup done in {time.time() - T0:.0f}s")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# F2a — explanation methods (all on the v25 ConvNeXt-S specialists; hook-free)
# ════════════════════════════════════════════════════════════════════════════════
import copy, cv2, time as _time
import torch.nn.functional as _F
from scipy.stats import spearmanr, wilcoxon
from sklearn.metrics import roc_auc_score
from torchvision.transforms import GaussianBlur
F2_OUT = Path("/kaggle/working/f2_out"); F2_OUT.mkdir(parents=True, exist_ok=True)
F2 = {"meta": {"table": E_PREV_TABLE, "coverage": _cov, "moved_roots": _moved}, "faith": {}, "loc": {}, "trust": {}, "timing_ms": {}}
def f2_save(): (F2_OUT / "f2_results.json").write_text(json.dumps(F2, indent=1, default=float))

N_XAI, N_LOC, N_TRUST = 40, 100, 150
RISE_N, SCORE_K, IG_STEPS = 800, 64, 32
FRACS = np.array([0, .02, .05, .1, .2, .35, .5, .75, 1.0]); _blur = GaussianBlur(31, 10.0)
METHODS = ["Grad-CAM", "Grad-CAM++", "Grad-CAM++ (thesis code)", "LayerCAM (stage 3)", "EigenCAM",
           "Score-CAM", "Integrated Gradients", "RISE", "random"]
_AMP = torch.cuda.is_available()
F2["meta"].update(methods=METHODS, n_xai=N_XAI, n_loc=N_LOC, n_trust=N_TRUST, rise_n=RISE_N, score_k=SCORE_K, ig_steps=IG_STEPS)

def _feats(model, x):
    """stage-3 and final feature maps of a timm ConvNeXt (forward_features = stem → stages → norm_pre)."""
    bb = model.backbone; h = bb.stem(x); outs = []
    for st in bb.stages: h = st(h); outs.append(h)
    return outs[-2], bb.norm_pre(h)

def _logits(model, h): return model.head(model.backbone.forward_head(h))

def _nz(t): return torch.where(torch.isfinite(t), t, torch.zeros_like(t))

def grad_family(model, x, t):
    """one forward/backward → Grad-CAM, Grad-CAM++, thesis Grad-CAM++, LayerCAM(stage 3), EigenCAM; + final activations."""
    with torch.enable_grad():
        xi = x.detach().clone().requires_grad_(True)
        s3, h = _feats(model, xi); s3.retain_grad(); h.retain_grad()
        z = _logits(model, h); z[0, t].backward()
    A, g = h.detach()[0].float(), h.grad[0].float()
    A3, g3 = s3.detach()[0].float(), s3.grad[0].float()
    out = {"Grad-CAM": torch.relu((g.mean((1, 2))[:, None, None] * A).sum(0))}
    gp2, gp3 = g ** 2, g ** 3
    a = gp2 / (2 * gp2 + A.sum((1, 2))[:, None, None] * gp3 + 1e-7)                 # Chattopadhay et al. 2018
    a = _nz(torch.where(g != 0, a, torch.zeros_like(a)))
    out["Grad-CAM++"] = torch.relu(((a * torch.relu(g)).sum((1, 2))[:, None, None] * A).sum(0))
    gt = g * 100.0; gt2, gt3 = gt ** 2, gt ** 3                                         # thesis: score×100, Σ(A·g³)
    at = _nz(gt2 / (2 * gt2 + (A * gt3).sum((1, 2), keepdim=True) + 1e-8))
    out["Grad-CAM++ (thesis code)"] = torch.relu(((at * torch.relu(gt)).sum((1, 2))[:, None, None] * A).sum(0))
    out["LayerCAM (stage 3)"] = torch.relu((torch.relu(g3) * A3).sum(0))
    M = A.reshape(A.shape[0], -1).T; M = M - M.mean(0, keepdim=True)
    try:
        _, _, Vh = torch.linalg.svd(M, full_matrices=False); e = (M @ Vh[0]).reshape(A.shape[1:])
        if torch.corrcoef(torch.stack([e.flatten(), A.mean(0).flatten()]))[0, 1] < 0: e = -e   # SVD sign is arbitrary
        out["EigenCAM"] = torch.relu(e)
    except Exception:
        out["EigenCAM"] = torch.zeros_like(A[0])
    return {k: v.cpu().numpy() for k, v in out.items()}, A

@torch.no_grad()
def score_cam(model, x, t, A):
    k = min(SCORE_K, A.shape[0]); idx = A.mean((1, 2)).topk(k).indices
    M = _F.interpolate(A[idx][None], size=x.shape[2:], mode="bilinear", align_corners=False)[0]
    lo, hi = M.flatten(1).min(1)[0][:, None, None], M.flatten(1).max(1)[0][:, None, None]
    M = (M - lo) / (hi - lo + 1e-8); ps = []
    for i in range(0, k, 32):
        with torch.cuda.amp.autocast(enabled=_AMP):
            ps.append(torch.softmax(model(x * M[i:i + 32, None]).float(), 1)[:, t])
    return torch.relu((torch.cat(ps)[:, None, None] * A[idx]).sum(0)).cpu().numpy()

def integrated_gradients(model, x, t):
    base = _blur(x); diff = x - base; tot = torch.zeros_like(x)
    al = torch.linspace(1.0 / IG_STEPS, 1.0, IG_STEPS, device=x.device)
    with torch.enable_grad():
        for i in range(0, IG_STEPS, 16):
            xi = (base + al[i:i + 16][:, None, None, None] * diff).detach().requires_grad_(True)
            model(xi)[:, t].sum().backward(); tot += xi.grad.sum(0, keepdim=True)
    attr = (diff * tot / IG_STEPS)[0].abs().sum(0).float().cpu().numpy()
    return cv2.GaussianBlur(attr, (0, 0), 4)

_gen = torch.Generator().manual_seed(0)
def _rise_masks(n, s=7, p=0.5, size=224):
    cell = int(np.ceil(size / s)); up = (s + 1) * cell
    big = _F.interpolate((torch.rand(n, 1, s, s, generator=_gen) < p).float(), size=(up, up), mode="bilinear", align_corners=False)
    off = torch.randint(0, cell, (n, 2), generator=_gen)
    return torch.stack([big[i, :, off[i, 0]:off[i, 0] + size, off[i, 1]:off[i, 1] + size] for i in range(n)]).half()
RISE_M = _rise_masks(RISE_N).to(DEVICE)

@torch.no_grad()
def rise(model, x, t):
    sal = torch.zeros(x.shape[2:], device=x.device)
    for i in range(0, RISE_N, 100):
        m = RISE_M[i:i + 100].float()
        with torch.cuda.amp.autocast(enabled=_AMP):
            p = torch.softmax(model(x * m).float(), 1)[:, t]
        sal += (p[:, None, None] * m[:, 0]).sum(0)
    return (sal / (RISE_N * 0.5)).cpu().numpy()

TIMING = {m: [] for m in METHODS}
def all_maps(model, x, t, timed=False):
    t0 = _time.time(); maps, A = grad_family(model, x, t); dt = (_time.time() - t0) / 5
    if timed:
        for k in maps: TIMING[k].append(dt)
    for name, fn in [("Score-CAM", lambda: score_cam(model, x, t, A)), ("Integrated Gradients", lambda: integrated_gradients(model, x, t)),
                     ("RISE", lambda: rise(model, x, t))]:
        t0 = _time.time(); maps[name] = fn()
        if timed: TIMING[name].append(_time.time() - t0)
    maps["random"] = np.random.default_rng(int(1e6 * (_time.time() % 1))).random((x.shape[2], x.shape[3])).astype(np.float32)
    return maps

def _up(m, hw): return cv2.resize(np.asarray(m, np.float32), (hw[1], hw[0]), interpolation=cv2.INTER_LINEAR)

@torch.no_grad()
def _curve(model, x, rank, target, mode):
    base = _blur(x); H, W = x.shape[2:]; xs = []
    for fr in FRACS:
        k = int(fr * H * W); m = torch.zeros(H * W, device=x.device); m[rank[:k]] = 1; m = m.view(1, 1, H, W)
        xs.append(x * (1 - m) + base * m if mode == "del" else base * (1 - m) + x * m)
    return torch.softmax(model(torch.cat(xs)).float(), 1)[:, target].cpu().numpy()

def _auc(y): return float(np.trapezoid(y, FRACS) if hasattr(np, "trapezoid") else np.trapz(y, FRACS))

def randomised(model):
    m = copy.deepcopy(model)
    for mod in list(m.head.modules()) + list(m.backbone.stages[-1].modules()):
        if isinstance(mod, (nn.Linear, nn.Conv2d)):
            nn.init.normal_(mod.weight, std=0.02)
            if mod.bias is not None: nn.init.zeros_(mod.bias)
    return m.eval()

def _rho(a, b):
    a, b = _up(a, (56, 56)).ravel(), _up(b, (56, 56)).ravel()
    return float(abs(spearmanr(a, b).correlation)) if a.std() > 0 and b.std() > 0 else 0.0

def _wil(a, b, alt):
    try: return float(wilcoxon(a, b, alternative=alt).pvalue)
    except Exception: return 1.0

def _prep(model):
    for p in model.parameters(): p.requires_grad_(False)
    return model.to(DEVICE).eval()

# ---- self-check: the hook-free forward reproduces model(x) ------------------------------------------
_o = next(iter(NEW_SPEC)); _m = _prep(NEW_SPEC[_o]["model"]); _x = torch.randn(1, 3, 224, 224, device=DEVICE)
with torch.no_grad(): _d = (_logits(_m, _feats(_m, _x)[1]) - _m(_x)).abs().max().item()
assert _d < 1e-3, f"hook-free forward differs from model(x): {_d}"
NEW_SPEC[_o]["model"] = _m.cpu(); print(f"✅ hook-free forward check: max |Δlogit| = {_d:.2e}")

# ════════════════════════════════════════════════════════════════════════════════
# F2b — faithfulness + sanity, 40 test images per organ (same images as Stage F)
# ════════════════════════════════════════════════════════════════════════════════
EXAMPLES = {}
print(f"  {'organ':<11}{'method':<26}{'del':>7}{'del rnd':>8}{'ins':>7}{'ins rnd':>8}{'p del':>8}{'p ins':>8}{'|ρ|':>6}")
for organ in E_ORGANS:
    if organ not in NEW_SPEC: continue
    if e_elapsed_h() > TIME_BUDGET_H - 2.0: print("⏱ time budget: remaining organs skipped"); break
    try:
        model = _prep(NEW_SPEC[organ]["model"]); rmodel = randomised(model)
        d = first_existing(TAB[(TAB.organ == organ) & (TAB.split == "test") & TAB.spec_ok].sample(frac=1, random_state=1), N_XAI)
        if len(d) < min(5, N_XAI): print(f"  ⚠️ {organ}: only {len(d)} readable test images — skipped"); continue
        R = {m: {"del": [], "ins": [], "rho": []} for m in METHODS}
        for j, p in enumerate(d.path):
            img = eval_img(load_rgb(p)); x = to_tensor(img).unsqueeze(0).to(DEVICE)
            with torch.no_grad(): t = int(model(x).argmax(1))
            maps = all_maps(model, x, t, timed=True); rmaps = all_maps(rmodel, x, t)
            if j == 0: EXAMPLES[organ] = (img, {k: _up(v, (224, 224)) for k, v in maps.items()})
            for m in METHODS:
                rank = torch.as_tensor(np.argsort(-_up(maps[m], (224, 224)).ravel()).copy(), device=DEVICE)
                R[m]["del"].append(_auc(_curve(model, x, rank, t, "del"))); R[m]["ins"].append(_auc(_curve(model, x, rank, t, "ins")))
                if m != "random": R[m]["rho"].append(_rho(maps[m], rmaps[m]))
        rd, ri = np.array(R["random"]["del"]), np.array(R["random"]["ins"]); F2["faith"][organ] = {}
        for m in METHODS:
            dl, il = np.array(R[m]["del"]), np.array(R[m]["ins"])
            r = {"n": len(dl), "del": float(dl.mean()), "ins": float(il.mean()), "del_random": float(rd.mean()), "ins_random": float(ri.mean()),
                 "p_del": _wil(dl, rd, "less") if m != "random" else None, "p_ins": _wil(il, ri, "greater") if m != "random" else None,
                 "win_del": float((dl < rd).mean()), "win_ins": float((il > ri).mean()),
                 "score": float((il - dl).mean() - (ri - rd).mean()), "sanity_rho": float(np.mean(R[m]["rho"])) if R[m]["rho"] else None}
            F2["faith"][organ][m] = r
            if m != "random":
                print(f"  {organ:<11}{m:<26}{r['del']:>7.3f}{r['del_random']:>8.3f}{r['ins']:>7.3f}{r['ins_random']:>8.3f}"
                      f"{r['p_del']:>8.3f}{r['p_ins']:>8.3f}{r['sanity_rho']:>6.2f}", flush=True)
        NEW_SPEC[organ]["model"] = model.cpu(); del rmodel; free_mem(f"F2 {organ}")
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"  ⚠️ F2 {organ}: {type(e).__name__}: {e}")
        NEW_SPEC[organ]["model"] = NEW_SPEC[organ]["model"].cpu()
    F2["timing_ms"] = {m: 1000 * float(np.median(v)) for m, v in TIMING.items() if v}; f2_save()

# ════════════════════════════════════════════════════════════════════════════════
# F2c — localisation on lesion masks, every method
# ════════════════════════════════════════════════════════════════════════════════

def dice(a, b): a, b = a.astype(bool), b.astype(bool); return float(2 * (a & b).sum() / (a.sum() + b.sum() + 1e-8))
def mask_box(m, pad=0.05):
    ys, xs = np.where(m)
    if len(xs) == 0: return None
    h, w = m.shape; px, py = pad * (xs.max() - xs.min() + 1), pad * (ys.max() - ys.min() + 1)
    return [max(0, xs.min() - px), max(0, ys.min() - py), min(w - 1, xs.max() + px), min(h - 1, ys.max() + py)]
def _largest(cm):
    n, lab, st, _ = cv2.connectedComponentsWithStats(cm.astype(np.uint8))
    return (lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])) if n > 1 else cm.astype(bool)

MEDSAM = None
try:
    from transformers import SamModel, SamProcessor
    MEDSAM = (SamModel.from_pretrained("wanglab/medsam-vit-base").to(DEVICE).eval(), SamProcessor.from_pretrained("wanglab/medsam-vit-base"))
except Exception as e: print("   MedSAM unavailable:", type(e).__name__, e)

@torch.no_grad()
def medsam_box(img_rgb, box):
    sam, proc = MEDSAM
    inp = proc(Image.fromarray(img_rgb), input_boxes=[[list(map(float, box))]], return_tensors="pt").to(DEVICE)
    out = sam(**inp, multimask_output=False)
    return proc.image_processor.post_process_masks(out.pred_masks.cpu(), inp["original_sizes"].cpu(), inp["reshaped_input_sizes"].cpu())[0][0, 0].numpy().astype(bool)

LOC_M = [m for m in METHODS if m != "random"] + ["random"]
for organ, pairs in seg_pairs().items():
    if organ not in NEW_SPEC or not pairs: continue
    if e_elapsed_h() > TIME_BUDGET_H - 1.0: print("⏱ time budget: localisation stopped"); break
    model = _prep(NEW_SPEC[organ]["model"]); cls = E_CLASSES[organ]; mal = MALIGNANT_CLASSES.get(organ, set())
    tgt_default = next((cls.index(c) for c in cls if c in mal), None)
    L = {m: {"dice": [], "point": [], "energy": [], "medsam": []} for m in LOC_M}; chance, gtbox, n_err = [], [], 0
    for ip, mp in pairs[:N_LOC]:
        try:
            img = np.array(Image.open(ip).convert("RGB")); gt = np.array(Image.open(mp).convert("L")) > 127
            if gt.shape != img.shape[:2]: gt = cv2.resize(gt.astype(np.uint8), img.shape[1::-1], interpolation=cv2.INTER_NEAREST) > 0
            if gt.sum() < 20: continue
            x = to_tensor(eval_img(img)).unsqueeze(0).to(DEVICE)
            with torch.no_grad(): pred = int(model(x).argmax(1))
            t = pred if (tgt_default is None or cls[pred] in mal) else tgt_default
            maps = all_maps(model, x, t); chance.append(float(gt.mean()))
            if MEDSAM is not None: gtbox.append(dice(medsam_box(img, mask_box(gt)), gt))
            for m in LOC_M:
                c = _up(maps[m], img.shape[:2]); c = (c - c.min()) / (c.max() - c.min() + 1e-8)
                cm = _largest(c >= 0.5); L[m]["dice"].append(dice(cm, gt))
                yy, xx = np.unravel_index(c.argmax(), c.shape); L[m]["point"].append(bool(gt[yy, xx]))
                L[m]["energy"].append(float((c * gt).sum() / (c.sum() + 1e-8)))
                if MEDSAM is not None:
                    b = mask_box(cm, 0.10); L[m]["medsam"].append(dice(medsam_box(img, b), gt) if b else 0.0)
        except Exception as e:
            n_err += 1
            if n_err <= 2: print(f"   {organ}: {type(e).__name__}: {e}")
    if chance:
        F2["loc"][organ] = {"n": len(chance), "errors": n_err, "chance_energy": float(np.mean(chance)),
                            "gt_box_medsam": float(np.mean(gtbox)) if gtbox else None,
                            "methods": {m: {k: (float(np.mean(v)) if v else None) for k, v in L[m].items()} for m in LOC_M}}
        print(f"  {organ:<11} n={len(chance):3d}  chance energy={np.mean(chance):.3f}  GT-box MedSAM={np.mean(gtbox) if gtbox else float('nan'):.3f}")
        for m in LOC_M:
            r = F2["loc"][organ]["methods"][m]
            print(f"     {m:<26} Dice={r['dice']:.3f}  pointing={r['point']:.3f}  energy={r['energy']:.3f}  box→MedSAM={r['medsam'] if r['medsam'] is not None else float('nan'):.3f}", flush=True)
    NEW_SPEC[organ]["model"] = model.cpu(); free_mem(f"F2 loc {organ}"); f2_save()

# ════════════════════════════════════════════════════════════════════════════════
# F2d — trust signals: do they flag the specialist's OWN mistakes? (AUROC, higher = better)
#   msp = calibrated max-probability · entropy · TTA disagreement · MC-dropout (head) std ·
#   CAM focus (Grad-CAM mass in its top-10 % pixels; thesis "focus ratio") ·
#   trust proxy = mean rank of (1−msp, MC-dropout, TTA, 1−focus) — the thesis-style composite
# ════════════════════════════════════════════════════════════════════════════════
def _rankpct(v): v = np.asarray(v, float); return (np.argsort(np.argsort(v)) + 0.5) / len(v)

def trust_signals(organ, rows, bs=32):
    s = NEW_SPEC[organ]; model = _prep(s["model"]); T = s["T"]; out = {k: [] for k in ["msp", "entropy", "tta", "mc", "focus", "pred"]}
    drops = [mod for mod in model.head.modules() if isinstance(mod, nn.Dropout)]
    for i in range(0, len(rows), bs):
        x = torch.stack([to_tensor(eval_img(load_rgb(p))) for p in rows.path.iloc[i:i + bs]]).to(DEVICE)
        with torch.no_grad():
            P = torch.softmax(model(x).float() / T, 1); pred = P.argmax(1)
            Pa = [torch.softmax(model(a).float() / T, 1) for a in (torch.flip(x, [3]), torch.flip(x, [2]), torch.rot90(x, 1, [2, 3]))]
            tta = torch.stack([0.5 * (q - P).abs().sum(1) for q in Pa]).mean(0)
            f = model.backbone(x)
            for dm in drops: dm.train()
            mc = torch.stack([torch.softmax(model.head(f).float() / T, 1).gather(1, pred[:, None])[:, 0] for _ in range(20)]).std(0)
            for dm in drops: dm.eval()
        with torch.enable_grad():
            xi = x.detach().clone().requires_grad_(True); _, h = _feats(model, xi); h.retain_grad()
            _logits(model, h).gather(1, pred[:, None]).sum().backward()
            cam = torch.relu((h.grad.mean((2, 3), keepdim=True) * h.detach()).sum(1)).flatten(1).float()
        k = max(1, int(0.1 * cam.shape[1])); foc = cam.topk(k, 1).values.sum(1) / (cam.sum(1) + 1e-8)
        out["msp"] += P.max(1).values.cpu().tolist(); out["entropy"] += (-(P * (P + 1e-12).log()).sum(1)).cpu().tolist()
        out["tta"] += tta.cpu().tolist(); out["mc"] += mc.cpu().tolist(); out["focus"] += foc.cpu().tolist(); out["pred"] += pred.cpu().tolist()
    s["model"] = model.cpu(); return {k: np.array(v) for k, v in out.items()}

TEt = TAB[(TAB.split == "test") & TAB.spec_ok.astype(bool) & TAB.label_set.fillna("").astype(str).str.len().gt(0)]
POOL = {k: [] for k in ["msp", "entropy", "tta", "mc", "focus", "guard", "err", "organ"]}
for organ in E_ORGANS:
    if organ not in NEW_SPEC: continue
    if e_elapsed_h() > TIME_BUDGET_H - 0.5: print("⏱ time budget: trust stopped"); break
    try:
        To = TEt[TEt.organ == organ]
        if len(To) < 10: continue
        rows = per_group_existing(To, max(20, N_TRUST // max(1, To.source.nunique())), 7).reset_index(drop=True)
        if len(rows) < 10: print(f"  ⚠️ trust {organ}: only {len(rows)} readable images — skipped"); continue
        S = trust_signals(organ, rows)
        sets = _sets_of(rows, organ)
        err = np.array([not (st and int(pr) in {int(c) for c in st.split(",")}) for pr, st in zip(S["pred"], sets)])
        guard = np.full(len(rows), np.nan)      # gateway guard is evaluated in Stage F; not loaded here
        for k in ["msp", "entropy", "tta", "mc", "focus"]: POOL[k] += S[k].tolist()
        POOL["guard"] += guard.tolist(); POOL["err"] += err.tolist(); POOL["organ"] += [organ] * len(rows)
        F2["trust"][organ] = {"n": int(len(rows)), "errors": int(err.sum()), "sources": sorted(rows.source.unique().tolist())}
        print(f"  {organ:<11} n={len(rows):4d}  errors={int(err.sum()):3d}  sources={F2['trust'][organ]['sources']}", flush=True)
    except Exception as e:
        import traceback; traceback.print_exc(); print(f"  ⚠️ trust {organ}: {type(e).__name__}: {e}")

def _auroc(score, err):
    ok = np.isfinite(score)
    return float(roc_auc_score(err[ok], score[ok])) if ok.sum() > 10 and 0 < err[ok].sum() < ok.sum() else None
def _risk_at(score, err, cov=0.8):                         # error rate among the `cov` most-trusted (lowest score) images
    ok = np.isfinite(score); s, e = score[ok], err[ok]; keep = np.argsort(s)[: int(cov * len(s))]
    return float(e[keep].mean()) if len(keep) else None

if POOL["err"]:
    err = np.array(POOL["err"], bool); P = {k: np.array(POOL[k], float) for k in ["msp", "entropy", "tta", "mc", "focus", "guard"]}
    SIG = {"1 − max prob (calibrated)": 1 - P["msp"], "entropy": P["entropy"], "TTA disagreement": P["tta"],
           "MC-dropout std": P["mc"], "1 − CAM focus": 1 - P["focus"]}
    SIG["trust proxy (thesis-style composite)"] = np.mean([_rankpct(1 - P["msp"]), _rankpct(P["mc"]), _rankpct(P["tta"]), _rankpct(1 - P["focus"])], 0)
    F2["trust"]["_pooled"] = {"n": int(len(err)), "errors": int(err.sum()), "error_rate": float(err.mean()),
                              "signals": {k: {"auroc": _auroc(v, err), "risk_at_80": _risk_at(v, err)} for k, v in SIG.items()}}
    org = np.array(POOL["organ"]); F2["trust"]["_per_organ_auroc_msp"] = {
        o: _auroc(SIG["1 − max prob (calibrated)"][org == o], err[org == o]) for o in sorted(set(org)) if 5 <= err[org == o].sum() <= (org == o).sum() - 5}
    print(f"\n  trust signals — pooled n={len(err)}, errors={int(err.sum())} ({err.mean():.3f})")
    for k, v in F2["trust"]["_pooled"]["signals"].items():
        print(f"     {k:<40} AUROC={v['auroc'] if v['auroc'] is not None else float('nan'):.3f}   error rate @80% coverage={v['risk_at_80'] if v['risk_at_80'] is not None else float('nan'):.3f}")
f2_save(); free_mem("F2")


In [ ]:
try:
    # ════════════════════════════════════════════════════════════════════════════════
    # F2e — tables + figures
    # ════════════════════════════════════════════════════════════════════════════════
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    def tex(s): return str(s).replace("_", r"\_").replace("&", r"\&").replace("−", "$-$").replace("→", r"$\to$")
    def fmt(v, nd=3): return "--" if v is None or (isinstance(v, float) and not np.isfinite(v)) else f"{v:.{nd}f}"
    ORG = [o for o in E_ORGANS if o in F2["faith"]]; MM = [m for m in METHODS if m != "random"]

    # --- method summary over organs
    rows = []
    for m in MM:
        rr = [F2["faith"][o][m] for o in ORG]
        rows.append({"m": m, "dDel": np.mean([r["del_random"] - r["del"] for r in rr]), "dIns": np.mean([r["ins"] - r["ins_random"] for r in rr]),
                     "sigDel": sum(r["p_del"] < 0.05 for r in rr), "sigIns": sum(r["p_ins"] < 0.05 for r in rr),
                     "both": sum(r["p_del"] < 0.05 and r["p_ins"] < 0.05 for r in rr),
                     "rho": np.mean([r["sanity_rho"] for r in rr]), "ms": F2["timing_ms"].get(m)})
    F2["summary"] = rows
    L = [r"\begin{tabular}{@{}lccccccr@{}}", r"\toprule",
         r"Method & $\Delta$Del.$\uparrow$ & $\Delta$Ins.$\uparrow$ & \#Del. & \#Ins. & \#Both & Sanity $|\rho|\downarrow$ & ms/img \\",
         r"\midrule"]
    for r in sorted(rows, key=lambda r: -(r["dDel"] + r["dIns"])):
        L.append(f"{tex(r['m'])} & {r['dDel']:+.3f} & {r['dIns']:+.3f} & {r['sigDel']}/{len(ORG)} & {r['sigIns']}/{len(ORG)} & {r['both']}/{len(ORG)} & {r['rho']:.2f} & {fmt(r['ms'], 0)} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]; (F2_OUT / "tab_xai_methods.tex").write_text("\n".join(L))

    # --- best method per organ (sanity |ρ| < 0.5 required)
    L = [r"\begin{tabular}{@{}llcccc@{}}", r"\toprule", r"Organ & Best method & Score & Grad-CAM score & Del. (best/rand) & Ins. (best/rand) \\", r"\midrule"]
    F2["best"] = {}
    for o in ORG:
        cand = [(F2["faith"][o][m]["score"], m) for m in MM if (F2["faith"][o][m]["sanity_rho"] or 0) < 0.5]
        if not cand: continue
        s, m = max(cand); r = F2["faith"][o][m]; F2["best"][o] = m
        L.append(f"{tex(o)} & {tex(m)} & {s:+.3f} & {F2['faith'][o]['Grad-CAM']['score']:+.3f} & {r['del']:.3f}/{r['del_random']:.3f} & {r['ins']:.3f}/{r['ins_random']:.3f} \\\\")
    L += [r"\bottomrule", r"\end{tabular}"]; (F2_OUT / "tab_xai_best.tex").write_text("\n".join(L))

    # --- localisation, averaged over the masked organs
    if F2["loc"]:
        LO = list(F2["loc"]); L = [r"\begin{tabular}{@{}lcccc@{}}", r"\toprule", r"Method & Dice & Pointing & Energy in mask & Box$\to$MedSAM \\", r"\midrule"]
        for m in LOC_M:
            v = {k: np.mean([F2["loc"][o]["methods"][m][k] for o in LO if F2["loc"][o]["methods"][m][k] is not None] or [np.nan]) for k in ["dice", "point", "energy", "medsam"]}
            L.append(f"{tex(m)} & {fmt(v['dice'])} & {fmt(v['point'])} & {fmt(v['energy'])} & {fmt(v['medsam'])} \\\\")
        ch = np.mean([F2["loc"][o]["chance_energy"] for o in LO]); gb = [F2["loc"][o]["gt_box_medsam"] for o in LO if F2["loc"][o]["gt_box_medsam"] is not None]
        L += [r"\midrule", f"chance (lesion area) & -- & -- & {ch:.3f} & -- \\\\", f"GT box $\\to$ MedSAM (upper bound) & -- & -- & -- & {fmt(np.mean(gb) if gb else None)} \\\\",
              r"\bottomrule", r"\end{tabular}"]
        (F2_OUT / "tab_loc_methods.tex").write_text("\n".join(L))

    # --- trust signals
    if "_pooled" in F2["trust"]:
        tp = F2["trust"]["_pooled"]; L = [r"\begin{tabular}{@{}lcc@{}}", r"\toprule", r"Signal & AUROC (error) & Error rate @80\% coverage \\", r"\midrule"]
        for k, v in sorted(tp["signals"].items(), key=lambda kv: -(kv[1]["auroc"] or 0)):
            L.append(f"{tex(k)} & {fmt(v['auroc'])} & {fmt(v['risk_at_80'])} \\\\")
        L += [r"\midrule", f"all images (no rejection) & -- & {tp['error_rate']:.3f} \\\\", r"\bottomrule", r"\end{tabular}"]
        (F2_OUT / "tab_trust.tex").write_text("\n".join(L))

    # --- figure: score heat-map organs × methods
    if ORG:
        Mx = np.array([[F2["faith"][o][m]["score"] for m in MM] for o in ORG])
        fig, ax = plt.subplots(figsize=(1.0 * len(MM) + 2.5, 0.42 * len(ORG) + 1.8))
        lim = np.nanmax(np.abs(Mx)) or 1; im = ax.imshow(Mx, cmap="RdBu", vmin=-lim, vmax=lim, aspect="auto")
        for i in range(len(ORG)):
            for j in range(len(MM)):
                r = F2["faith"][ORG[i]][MM[j]]; star = "*" if (r["p_del"] < 0.05 and r["p_ins"] < 0.05) else ""
                ax.text(j, i, f"{Mx[i, j]:+.2f}{star}", ha="center", va="center", fontsize=7, color="white" if abs(Mx[i, j]) > 0.6 * lim else "black")
        ax.set_xticks(range(len(MM))); ax.set_xticklabels(MM, rotation=35, ha="right", fontsize=8)
        ax.set_yticks(range(len(ORG))); ax.set_yticklabels(ORG, fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.03, label="faithfulness vs random")
        ax.set_title("Faithfulness vs random ranking  (* = deletion and insertion both p<0.05)", fontsize=9)
        plt.tight_layout(); plt.savefig(F2_OUT / "fig_xai_methods.png", dpi=150, bbox_inches="tight"); plt.close()

    # --- figure: example maps
    ex = ([o for o in ["brain_2d", "colon", "skin", "kidney", "breast", "thyroid"] if o in EXAMPLES] or list(EXAMPLES))[:4]
    if ex:
        fig, ax = plt.subplots(len(ex), len(MM) + 1, figsize=(1.55 * (len(MM) + 1), 1.7 * len(ex)))
        ax = np.atleast_2d(ax)
        for i, o in enumerate(ex):
            img, maps = EXAMPLES[o]; ax[i, 0].imshow(img); ax[i, 0].set_title(o, fontsize=7)
            for j, m in enumerate(MM, 1):
                c = maps[m]; c = (c - c.min()) / (c.max() - c.min() + 1e-8)
                ax[i, j].imshow(img); ax[i, j].imshow(c, cmap="jet", alpha=0.45)
                if i == 0: ax[i, j].set_title(m.replace(" (", "\n("), fontsize=6)
        for a in ax.ravel(): a.axis("off")
        plt.tight_layout(); plt.savefig(F2_OUT / "fig_xai_methods_examples.png", dpi=120, bbox_inches="tight"); plt.close()

    # --- figure: trust AUROC
    if "_pooled" in F2["trust"]:
        it = [(k, v["auroc"]) for k, v in F2["trust"]["_pooled"]["signals"].items() if v["auroc"] is not None]
        it.sort(key=lambda kv: kv[1])
        fig, ax = plt.subplots(figsize=(6.5, 0.45 * len(it) + 1.2))
        ax.barh([k for k, _ in it], [v for _, v in it], color="#2f6db3"); ax.axvline(0.5, color="k", lw=0.8, ls="--")
        ax.set_xlim(0.3, 1.0); ax.set_xlabel("AUROC for detecting the specialist's own errors (0.5 = chance)")
        plt.tight_layout(); plt.savefig(F2_OUT / "fig_trust_auroc.png", dpi=150, bbox_inches="tight"); plt.close()
    f2_save()
    print("✅ tables:", sorted(p.name for p in F2_OUT.glob("*.tex")), "\n✅ figures:", sorted(p.name for p in F2_OUT.glob("*.png")))

except Exception as _e:
    import traceback; traceback.print_exc()
    print('⚠️ F2 figures failed — skipped:', type(_e).__name__, _e)
    try: f2_save()
    except Exception: pass


In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# F2f — package
# ════════════════════════════════════════════════════════════════════════════════
import shutil
F2["meta"]["elapsed_h"] = e_elapsed_h(); f2_save()
z = shutil.make_archive("/kaggle/working/mmaf_v25_F2", "zip", F2_OUT)
print(f"✅ {z}  ({Path(z).stat().st_size/1e6:.1f} MB) — elapsed {e_elapsed_h():.2f} h")
